In [ ]:
import pandas as pd
import numpy as np

# Load data

In [ ]:
df = pd.read_csv("outputs/2.0-discovery_results.csv")
df

# Initial field profiles

In [ ]:
from collections import Counter

def df_to_clipboard(df, index=True):
    print(df.to_markdown(index=index))


def top_n_values(lst, n, filter_keyword=None):
    if filter_keyword:
        lst = [x for x in lst if filter_keyword not in str(x).lower()]

    return [x for x, _ in Counter(lst).most_common(n)]

In [ ]:
snapshot_count = df["snapshot_id"].nunique()

profiles = (
    df.groupby("metadata_field")
    .agg(
        count=("metadata_field", "count"),
        n_snapshots=("snapshot_id", "nunique"),
        support=("snapshot_id", lambda x: x.nunique() / snapshot_count),
        corpora_count=("source", "nunique"),
        figure_count=("snapshot_type", lambda x: (x == "figure").sum()),
        table_count=("snapshot_type", lambda x: (x == "table").sum()),
        source_snapshot_count=("source_level", lambda x: (x == "snapshot").sum()),
        source_document_count=("source_level", lambda x: (x == "document").sum()),
        source_both_count=("source_level", lambda x: (x == "both").sum()),
        top_observed_values=(
            "observed_value",
            lambda x: top_n_values(x, 5, filter_keyword="not identifiable"),
        ),
        n_unique_observed_values=(
            "observed_value",
            lambda x: x[
                ~x.str.contains("not identifiable", case=False, na=False)
            ].nunique(),
        ),
        top_description_values=("description", lambda x: top_n_values(x, 3)),
        top_reasoning_values=("reasoning", lambda x: top_n_values(x, 3)),
    )
    .sort_values("count", ascending=False)
)

profiles

In [ ]:
# df_to_clipboard(profiles.head(10))

In [ ]:
profiles.to_csv("outputs/3.0-field_profiles.csv")